# 05 - SMOTENC (interpolation control)

Raises every class to `N_MAX` with SMOTENC. CPU only. Run once per dataset.
See `PIPELINE.md`, step 5.

In [ ]:
# Colab: upload df_train_<dataset>.csv.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import os

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, SMOTENC

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."

SEED = 42
K_NEIGHBORS = 5
MAX_DISCRETE_LEVELS = 12  # columns with at most this many levels are treated as nominal

np.random.seed(SEED)

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{DATASET}.csv"))
TARGET_COL = df_train.columns[-1]
FEATURES = [c for c in df_train.columns if c != TARGET_COL]
DISCRETE_COLS = [c for c in FEATURES if df_train[c].nunique() <= MAX_DISCRETE_LEVELS]

COUNTS = df_train[TARGET_COL].value_counts().sort_index()
N_MAX  = int(COUNTS.max())
print("train:", df_train.shape, "| class counts:", COUNTS.tolist())
print("discrete columns:", len(DISCRETE_COLS), "of", len(FEATURES))

## Resample

In [ ]:
X, y = df_train[FEATURES], df_train[TARGET_COL]
counts = y.value_counts()
strategy = {c: N_MAX for c in counts.index if counts[c] < N_MAX}
k = min(K_NEIGHBORS, int(counts.min()) - 1)
assert k >= 1, "a class has too few rows for SMOTE"

if DISCRETE_COLS:
    sampler = SMOTENC(categorical_features=[FEATURES.index(c) for c in DISCRETE_COLS],
                      sampling_strategy=strategy, k_neighbors=k, random_state=SEED)
else:                      # no nominal column (Page Blocks): SMOTENC reduces to SMOTE
    sampler = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=SEED)

X_res, y_res = sampler.fit_resample(X, y)
df_aug = pd.DataFrame(X_res, columns=FEATURES)
df_aug[TARGET_COL] = np.asarray(y_res)
df_aug = df_aug[df_train.columns]
print(f"rows: {len(df_train):,} -> {len(df_aug):,}")

## Export

In [ ]:
assert df_aug.isna().sum().sum() == 0
OUT_FILE = os.path.join(DATA_DIR, f"smotenc_{DATASET}_nmax{N_MAX}_seed{SEED}.csv")
df_aug.to_csv(OUT_FILE, index=False)
print("written:", OUT_FILE)

if IN_COLAB:
    files.download(OUT_FILE)